# Wall Floor Boundary YOLO Segmentation v1

X-AnyLabeling에서 `YOLO Segmentation` 형식으로 export한 벽-바닥 경계 dataset을 학습한다.

- class: `wall_floor_boundary`
- base model: `yolov8n-seg.pt`
- output run: `runs/segment/wall_floor_boundary_seg_v1`
- final robot model name: `wall_floor_boundary_seg_v1.pt`


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
ZIP = next(iter(uploaded))

EXTRACT = Path('/content/wallseg_upload')
shutil.rmtree(EXTRACT, ignore_errors=True)
EXTRACT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP, 'r') as z:
    z.extractall(EXTRACT)

print('uploaded:', ZIP)
print('top-level dirs:')
for p in sorted(EXTRACT.glob('*')):
    print(' ', p)


In [ ]:
from pathlib import Path
import shutil, random, yaml

EXTRACT = Path('/content/wallseg_upload')
OUT = Path('/content/wallseg_yolo')

def find_dataset_root(root: Path) -> Path:
    candidates = [p for p in root.rglob('*') if p.is_dir() and (p / 'images').is_dir() and (p / 'labels').is_dir()]
    if not candidates:
        raise FileNotFoundError('images/ and labels/ folders were not found. Export as YOLO Segmentation first.')
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

SRC = find_dataset_root(EXTRACT)
print('dataset root:', SRC)

image_exts = {'.jpg', '.jpeg', '.png'}
imgs = sorted([p for p in (SRC / 'images').iterdir() if p.suffix.lower() in image_exts])
pairs = []
missing = []
empty = []

for img in imgs:
    lbl = SRC / 'labels' / f'{img.stem}.txt'
    if not lbl.exists():
        missing.append(img.name)
        continue
    if lbl.stat().st_size == 0:
        empty.append(lbl.name)
        continue
    pairs.append((img, lbl))

print('images:', len(imgs))
print('labeled non-empty pairs:', len(pairs))
print('missing labels:', len(missing))
print('empty labels:', len(empty))
if len(pairs) < 5:
    raise ValueError('Need more labeled images before training. Label at least 20-30 for a first smoke test.')

shutil.rmtree(OUT, ignore_errors=True)
for split in ['train', 'val']:
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)

random.seed(0)
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.15))
val = pairs[:n_val]
train = pairs[n_val:]

def copy_pairs(items, split):
    for img, lbl in items:
        shutil.copy2(img, OUT / 'images' / split / img.name)
        shutil.copy2(lbl, OUT / 'labels' / split / lbl.name)

copy_pairs(train, 'train')
copy_pairs(val, 'val')

data = {
    'path': str(OUT),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'wall_floor_boundary'},
}
with open(OUT / 'data.yaml', 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print((OUT / 'data.yaml').read_text())
print('train:', len(train), 'val:', len(val))


In [ ]:
!pip -q install ultralytics

from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')
results = model.train(
    data='/content/wallseg_yolo/data.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    patience=25,
    name='wall_floor_boundary_seg_v1',
)


In [ ]:
from pathlib import Path
from google.colab import files
import shutil

best = Path('/content/runs/segment/wall_floor_boundary_seg_v1/weights/best.pt')
export = Path('/content/wall_floor_boundary_seg_v1.pt')
if not best.exists():
    raise FileNotFoundError(best)
shutil.copy2(best, export)
files.download(str(export))


로봇으로 복사:

```bash
scp wall_floor_boundary_seg_v1.pt seventt@10.42.0.1:/home/seventt/seventt/workspace/models/
```

주의: 기존 `models/wide.pt`, `models/cube.pt`는 덮어쓰지 않는다.
